# 머신러닝 기반 영화 리뷰 감성 분석
1. 데이터 준비 (전처리된 데이터 -> 특징 벡터 추출 (TF-IDF))
2. 머신러닝 모델별 학습 및 평가
3. 배포 준비 : 영화 리뷰 긍부정 판단 테스트 -> 클래스 생성, 모델 저장

## 1.데이터 준비 (전처리된 데이터)

In [1]:
import pandas as pd
data_filename = './data/Korean_movie_reviews_2016.csv'

# 데이터 로딩
review_df = pd.read_csv(data_filename)
review_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [2]:
review_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [3]:
# 입력 데이터와 정답 데이터 추출
print(type(review_df))
print(type(review_df.review))
review_list = list(review_df.review)
print(type(review_list))
label_list = list(review_df.label)


<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
<class 'list'>


In [4]:
from sklearn.model_selection import train_test_split
# 학습 데이터와 평가 데이터 분리

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.2, random_state=42)
print(test_X[:3])
print(test_y[:3])
print(len(train_X), len(train_y), len(test_X), len(test_y))

['예진 필모 역대 인생 연기', '어머니 모시 보기 좋 즉 년대 영화 수준 입니 애국심 등등 고려해도 객관 평점 정도 적당해 보입니', '조정석 멱살 잡고 끌 영화 막장드라마 요소 섞어 듯 느낌 근래 본 영화 최악']
[1, 0, 0]
132307 132307 33077 33077


## 2. 특징 벡터 추출 : TF-IDF

In [5]:
# 한국어 감성 분석용 tokenizer 정의
from konlpy.tag import Okt

def korean_tokenizer(text):
    my_tags= ['Noun', 'Adjective', 'Verb']
    my_stopwords = []

    return [word for word, tag in Okt().pos(text) if tag in my_tags and word not in my_stopwords]

In [6]:
# 최대 단어 수 1000개
# 학습 데이터로 Vectorizer 생성 및 학습 데이터 특징 벡터 추출
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    tokenizer=korean_tokenizer,
    max_features=1000
    ) 
vectorizer.fit(train_X)
print(vectorizer.get_feature_names_out()[:10])
len(vectorizer.get_feature_names_out())


c:\Users\user\anaconda3\envs\aiservice26\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


['가' '가고' '가는' '가볍' '가서' '가슴' '가장' '가족' '가지' '각본']


1000

In [7]:
# 학습데이터의 특징 벡터 추출
train_X_fv = vectorizer.transform(train_X)
print(train_X_fv.shape)
print(train_X_fv)


(132307, 1000)
  (np.int32(0), np.int32(83))	0.7137517875679297
  (np.int32(0), np.int32(398))	0.4020856619104301
  (np.int32(0), np.int32(770))	0.5734854019324609
  (np.int32(1), np.int32(90))	0.42502472668315505
  (np.int32(1), np.int32(398))	0.33552329552375876
  (np.int32(1), np.int32(588))	0.5034046813587029
  (np.int32(1), np.int32(644))	0.5106533836450027
  (np.int32(1), np.int32(757))	0.43885640980489105
  (np.int32(2), np.int32(625))	0.7270429956486876
  (np.int32(2), np.int32(775))	0.6865919330127485
  (np.int32(3), np.int32(90))	0.1509134471142437
  (np.int32(3), np.int32(97))	0.19407924332041185
  (np.int32(3), np.int32(106))	0.2229720867869912
  (np.int32(3), np.int32(108))	0.22785784660789868
  (np.int32(3), np.int32(121))	0.24473545058124246
  (np.int32(3), np.int32(126))	0.1699069003946714
  (np.int32(3), np.int32(130))	0.23624482096159452
  (np.int32(3), np.int32(283))	0.20911735290805225
  (np.int32(3), np.int32(308))	0.2518449018090405
  (np.int32(3), np.int32(373))	

In [9]:
train_X_fv.toarray()[3:4]

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

In [ ]:
train_X_fv.shape

(132307, 1000)

In [10]:
# 테스트 데이터 특징 벡터 추출
test_X_fv = vectorizer.transform(test_X)
print(test_X_fv)

  (np.int32(0), np.int32(612))	0.6776651431649237
  (np.int32(0), np.int32(617))	0.4276425354046395
  (np.int32(0), np.int32(713))	0.5982406001367304
  (np.int32(1), np.int32(162))	0.42323951691299944
  (np.int32(1), np.int32(374))	0.314467533439062
  (np.int32(1), np.int32(462))	0.3279949078011551
  (np.int32(1), np.int32(565))	0.42393758251364405
  (np.int32(1), np.int32(627))	0.12644756152872036
  (np.int32(1), np.int32(721))	0.2608429559291337
  (np.int32(1), np.int32(798))	0.2888019476049074
  (np.int32(1), np.int32(824))	0.20808949298054785
  (np.int32(1), np.int32(926))	0.24205604863103616
  (np.int32(1), np.int32(966))	0.40662978511471
  (np.int32(2), np.int32(101))	0.46166368033414257
  (np.int32(2), np.int32(180))	0.3118263763549253
  (np.int32(2), np.int32(239))	0.32418592279725533
  (np.int32(2), np.int32(390))	0.26545157684674564
  (np.int32(2), np.int32(627))	0.2615547797120316
  (np.int32(2), np.int32(653))	0.4264243026617292
  (np.int32(2), np.int32(814))	0.416965643067

In [11]:
# 정답 데이터 변환 (np.array)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)

In [13]:
# 데이터 일부 확인
print(train_y[:10])
print(test_y[:10])

[1 0 1 0 1 1 0 0 1 1]
[1 0 0 1 1 0 1 1 0 1]


## 3. 머신러닝 모델별 학습 및 평가
* 의사결정 트리
* 랜덤포레스트
* 나이브 베이즈
* 로지스틱 회귀 분석
* SVM
* Perceptron

In [21]:
# 머신러닝 모델별 학습 성능 평가 결과 저장 준비
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])

## 3.1 의사결정 트리 (Decision Tree)

In [14]:
# 학습
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [19]:
# 성능 평가
train_score = dtc.score(train_X_fv, train_y) * 100
print("학습 점수", train_score)
test_score = dtc.score(test_X_fv, test_y) * 100
print("검증 점수", test_score)


학습 점수 98.13690885591843
검증 점수 79.64144269431932


In [23]:
# 평가 결과 score_df에 추가
score_df.loc['DecisionTree'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.136909,79.641443


## 3.2 랜덤 포레스트 (Random Forrest)

In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_jobs= -1
)

rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [25]:
train_score = rf.score(train_X_fv, train_y) * 100
test_score = rf.score(test_X_fv, test_y) * 100
train_score, test_score

(98.1361530380101, 85.13770898207214)

In [26]:
score_df.loc['RandomForest'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.136909,79.641443
RandomForest,98.136153,85.137709


## 3.3 나이브 베이즈 (Naive Baysian)

In [27]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [29]:
train_score = mnb.score(train_X_fv, train_y) * 100
test_score = mnb.score(test_X_fv, test_y) * 100
print(train_score, test_score)

85.2668415125428 85.24049944069898


In [30]:
score_df.loc['NaiveBays'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.136909,79.641443
RandomForest,98.136153,85.137709
NaiveBays,85.266842,85.240499


## 3.4 로지스틱 회귀 분석 (Logistic Regression)

In [31]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)


LogisticRegression(solver='liblinear')

In [32]:
train_score = lr.score(train_X_fv, train_y) * 100
test_score = lr.score(test_X_fv, test_y) * 100
print(train_score, test_score)

86.17079973092882 86.05375336336427


In [33]:
score_df.loc['LogisticRegression'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.136909,79.641443
RandomForest,98.136153,85.137709
NaiveBays,85.266842,85.240499
LogisticRegression,86.170800,86.053753


## 3.5 SVM (Support Vector Machine)

In [34]:
from sklearn.svm import LinearSVC

svc = LinearSVC(
    verbose=True
)

svc.fit(train_X_fv, train_y)


[LibLinear]

LinearSVC(verbose=True)

In [35]:
train_score = svc.score(train_X_fv, train_y) * 100
test_score = svc.score(test_X_fv, test_y) * 100
print(train_score, test_score)

86.1398111966865 86.05073011458113


In [36]:
score_df.loc['SVM'] = [train_score, test_score]
score_df

,train,test
DecisionTree,98.136909,79.641443
RandomForest,98.136153,85.137709
NaiveBays,85.266842,85.240499
LogisticRegression,86.170800,86.053753
SVM,86.139811,86.050730


## 3.6 Perceptron

## 3.7 성능 비교

In [41]:
# 평가 결과 저장 데이터 프레임 확인
score_df.sort_values(by='test', ascending=False)

,train,test
LogisticRegression,86.170800,86.053753
SVM,86.139811,86.050730
NaiveBays,85.266842,85.240499
RandomForest,98.136153,85.137709
DecisionTree,98.136909,79.641443


## 4. 영화 리뷰 긍부정 판단
* 학습된 모델 중 선택하여 활용

In [52]:
# 학습 모델 선택
# sa_model = lr

# 영화 리뷰감성 분석용 tokenizer 정의
# korean_tokenizer
# 특징 벡터 추출 모델: vectorizer

sa_model = rf

In [55]:
review = '영화가 재미있다'

# 텍스트 전처리

# 특징 벡터 추출
review_fv = vectorizer.transform([review])
print(review_fv)

# 예측
pred = sa_model.predict(review_fv)
print(pred)
# 예측 결과 출력

result = '긍정' if pred >= 0.5 else '부정'
print(f'{review} -> {result}({pred})')

  (np.int32(0), np.int32(627))	0.28959553358902584
  (np.int32(0), np.int32(764))	0.9571491142582159
[1]
영화가 재미있다 -> 긍정([1])


In [53]:
# 함수로 만들기
def analyz_semtiment(review):
    # 1. 특징 벡터 추출
    review_fv = vectorizer.transform([review])
    # 2. 학습된 모델로 예측
    pred = sa_model.predict(review_fv)
    # 3. 예측값에 따라 결과를 생성하여 반환
    result = '긍정' if pred[0] >= 0.5 else '부정'

    return result, pred[0]

In [56]:
# 함수 테스트
reviews = [
    '이 영화 개꿀잼 ㅋㅋㅋ',
    '하품만 나온다',
    '이 영화 핵노잼 ㅠㅠ',
    '이딴게 영화냐 ㅉㅉ',
    '와 개쩐다',
    '감독 뭐하는 놈이냐',
    '정말 세계관 최강자들의 영화다'
]

for review in reviews:
    sentiment, prob = analyz_semtiment(review)
    print(f'{review} -> {sentiment}({prob})')


이 영화 개꿀잼 ㅋㅋㅋ -> 부정(0)
하품만 나온다 -> 긍정(1)
이 영화 핵노잼 ㅠㅠ -> 부정(0)
이딴게 영화냐 ㅉㅉ -> 부정(0)
와 개쩐다 -> 긍정(1)
감독 뭐하는 놈이냐 -> 부정(0)
정말 세계관 최강자들의 영화다 -> 긍정(1)


In [60]:
# 문장을 입력 받아서 긍부정 판단 
review = input('>> 리뷰 입력 : ')
sentiment, prob = analyz_semtiment(review)
print(f'{review} -> {sentiment}({prob})')


최근에 울어본 기억이 없는데 이 영화를 보고 진짜 펑펑 울었음 -> 긍정(1)


In [ ]:
# 배포를 위한 모델 저장

# 특징 추출용 vectorizer
import joblib

joblib.dump(vectorizer, './model/sa_movie_vectorizer.pkl')
joblib.dump(sa_model, './model/sa_movie_predict.pkl')

['./model/sa_movie_predict.pkl']

In [64]:
import joblib

def korean_tokenizer(text):
    my_tags= ['Noun', 'Adjective', 'Verb']
    my_stopwords = []

    return [word for word, tag in Okt().pos(text) if tag in my_tags and word not in my_stopwords]

class SentimentAnalyzer:
    def __init__(self, tokenizer,vectorizer_file, predict_model_file):
        self.vetorizer = joblib.load(vectorizer_file)
        self.predict_model = joblib.load(predict_model_file)
        self.tokenizer = tokenizer

    def analyz_semtiment(self, review):
        # 1. 특징 벡터 추출
        review_fv = self.__vectorizer.transform([review])
        # 2. 학습된 모델로 예측
        pred = self.__predict_model.predict(review_fv)
        # 3. 예측값에 따라 결과를 생성하여 반환
        result = '긍정' if pred[0] >= 0.5 else '부정'

        return result, pred[0]

